# Evaluación N.º 2 — Visualización de Datos de Transporte
## Operaciones de Aeronaves en aeródromos de Chile (JAC / DGAC), 2000–2026

**Pregunta principal.** ¿Cómo se distribuye y cómo ha evolucionado entre 2000 y 2026 la actividad de aterrizajes y despegues informada por la DGAC, según aeródromo, ámbito (doméstico / internacional) y tipo de actividad, y qué diferencias revela el shock de 2020?

| | Pregunta secundaria |
|---|---|
| **P1** | ¿Qué aeródromos concentran las operaciones en 2025 (último año completo) y qué proporción corresponde a tráfico internacional? |
| **P2** | ¿Cómo evolucionó el volumen mensual de operaciones entre 2000 y 2026, y existe un patrón estacional estable? |
| **P3** | ¿En qué se diferencian los tipos de actividad (pasajeros, instrucción, privada, carga…) en el tamaño de aeronave (PMD) y en su horario de operación? |

**H1 (exploratoria, no es conclusión).** Tras la caída de 2020, el tráfico internacional se recuperó más lentamente que el doméstico: tardó más años en volver al nivel de 2019.

La investigación de la fuente (Partes I y II) está en [`docs/01_investigacion_fuente.md`](../docs/01_investigacion_fuente.md).

**Estructura:** 1 Importaciones · 2 Configuración · 3 Carga · 4 Perfilamiento · 5 Calidad · 6 Limpieza · 7 Transformaciones · 8 Dimensionalidad · 9 Visualizaciones · 10 Benchmark · 11 Validación · 12 Interpretación · 13 Conclusiones

## 1. Importaciones

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pyarrow.parquet as pq

## 2. Configuración

El notebook se ejecuta desde `notebooks/`; la raíz del proyecto se agrega al `sys.path` para importar los módulos de `src/`.

In [2]:
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import preparacion as prep
import estilo as st

st.aplicar_estilo()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

RAW, PROCESSED, FIGURES = prep.RAW, prep.PROCESSED, st.FIGURES
print("pandas", pd.__version__, "| seaborn", sns.__version__)
print("Archivos originales:", sorted(p.name for p in RAW.iterdir() if p.is_file()))

pandas 3.0.6 | seaborn 0.13.2
Archivos originales: ['.gitkeep', 'bitacora-vuelos.parquet', 'dim_actividad_dgac.csv', 'dim_aerolinea.csv', 'dim_aeropuerto.csv', 'manifest.json', 'operaciones-aeropuertos.csv']


## 3. Carga

Se verifica primero la integridad de los archivos originales contra el manifiesto de procedencia (`data/raw/manifest.json`). Si falta el parquet (no se versiona por superar los 100 MB de GitHub) se descarga con `python src/download_data.py`.

In [3]:
import json, hashlib

manifest = json.loads((RAW / "manifest.json").read_text("utf-8"))
for nombre, meta in manifest.items():
    if nombre.startswith("_"):
        continue
    path = RAW / nombre
    if not path.exists():
        raise FileNotFoundError(f"Falta {nombre}: ejecute `python src/download_data.py`")
    h = hashlib.sha256(path.read_bytes()).hexdigest()
    estado = "OK" if h == meta["sha256"] else "VERSIÓN DISTINTA (la fuente se actualiza mensualmente)"
    print(f"{nombre:32s} {path.stat().st_size/1e6:8.2f} MB  {estado}")

bitacora-vuelos.parquet            138.25 MB  OK
operaciones-aeropuertos.csv          0.29 MB  OK
dim_aeropuerto.csv                   0.10 MB  OK
dim_aerolinea.csv                    0.01 MB  OK
dim_actividad_dgac.csv               0.00 MB  OK


Metadatos del parquet: se lee el esquema sin cargar los datos.

In [4]:
meta = pq.ParquetFile(RAW / "bitacora-vuelos.parquet")
print(meta.metadata)
print(meta.schema_arrow)

  created_by: Polars
  num_columns: 12
  num_rows: 11163791
  num_row_groups: 91
  format_version: 1.0
  serialized_size: 98251
dt_operacion: timestamp[us, tz=America/Santiago]
aeropuerto_oaci: large_string
tipo_operacion: large_string
aeropuerto_dgac_orig_dest: large_string
aerolinea_dgac: large_string
numero_vuelo: large_string
actividad_cod: large_string
matricula: large_string
modelo_avion: large_string
modelo_avion_desc: large_string
es_internacional: bool
pmd: decimal128(10, 5)


La bitácora se carga con las columnas de texto como **categóricas** (diccionario Arrow). Con `pd.read_parquet` por defecto ocupa ≈ 2,5 GB en memoria; como categóricas, ≈ 360 MB, sin pérdida de información.

In [5]:
t0 = time.perf_counter()
raw = prep.cargar_bitacora()
print(f"Carga: {time.perf_counter()-t0:.1f} s | memoria: {raw.memory_usage(deep=True).sum()/1e6:,.0f} MB")

resumen = pd.read_csv(prep.RESUMEN)
dim_aer = prep.cargar_dim_aeropuerto()
dim_act = prep.cargar_dim_actividad()

Carga: 7.7 s | memoria: 359 MB


### 3.1 Comparación de formatos (Parte VI)

**Los dos recursos publicados no son equivalentes**: el Parquet es la bitácora a nivel de operación (11,2 M filas, 12 columnas) y el CSV es un resumen mensual por aeródromo y ámbito (16 mil filas, 4 columnas, sin sobrevuelos). Comparar sus tiempos de carga directamente sería una comparación artificial: mediría el tamaño de los datos, no el formato.

Para una comparación justa se escribe **el mismo subconjunto** (operaciones de 2019 de la bitácora original, todas las columnas) en ambos formatos, en una carpeta temporal fuera del control de versiones, y se miden ambos con idéntico contenido.

In [6]:
bench_dir = PROCESSED / "_benchmark_formatos"
bench_dir.mkdir(parents=True, exist_ok=True)
y2019 = raw[raw["dt_operacion"].dt.year == 2019].copy()
for c in prep.TEXT_COLS:
    y2019[c] = y2019[c].astype("string")
y2019.to_csv(bench_dir / "bitacora_2019.csv", index=False)
y2019.to_parquet(bench_dir / "bitacora_2019.parquet", compression="zstd", index=False)

def medir(lector, path, n=3):
    tiempos = []
    for _ in range(n):
        t0 = time.perf_counter(); d = lector(path); tiempos.append(time.perf_counter() - t0)
    return d, min(tiempos)

d_csv, t_csv = medir(pd.read_csv, bench_dir / "bitacora_2019.csv")
d_pq, t_pq = medir(pd.read_parquet, bench_dir / "bitacora_2019.parquet")

comp_formatos = pd.DataFrame({
    "tamaño_archivo_MB": [(bench_dir / f"bitacora_2019.{e}").stat().st_size / 1e6 for e in ("csv", "parquet")],
    "tiempo_carga_s": [t_csv, t_pq],
    "memoria_MB": [d.memory_usage(deep=True).sum() / 1e6 for d in (d_csv, d_pq)],
    "filas": [len(d_csv), len(d_pq)],
}, index=["CSV", "Parquet (zstd)"])
comp_formatos

,tamaño_archivo_MB,tiempo_carga_s,memoria_MB,filas
CSV,41.65,1.34,77.40,528890
Parquet (zstd),6.40,0.06,63.70,528890


Tipos inferidos por cada lector sobre el mismo contenido:

In [7]:
pd.DataFrame({"CSV": d_csv.dtypes.astype(str), "Parquet": d_pq.dtypes.astype(str)})

,CSV,Parquet
dt_operacion,str,"datetime64[us, America/Santiago]"
aeropuerto_oaci,str,string
tipo_operacion,str,string
aeropuerto_dgac_orig_dest,str,string
aerolinea_dgac,str,string
numero_vuelo,str,string
actividad_cod,str,string
matricula,str,string
modelo_avion,str,string
modelo_avion_desc,str,string


In [8]:
del d_csv, d_pq, y2019

**¿Qué formato conviene?** Para la bitácora, **Parquet**: el archivo es varias veces más pequeño, carga más rápido y —lo más importante para la calidad— **conserva los tipos**: la fecha llega como `datetime` con zona horaria, `es_internacional` como booleano y `pmd` como numérico. El CSV entrega la fecha como texto (hay que volver a parsearla con `parse_dates` y tratar la zona horaria a mano) y el tipo de cada columna depende de la inferencia sobre su contenido: aquí `actividad_cod` y `numero_vuelo` quedaron como texto sólo porque mezclan letras y dígitos; un subconjunto con únicamente dígitos se habría leído como entero (perdiendo, por ejemplo, ceros a la izquierda). El CSV sólo es preferible para tablas pequeñas que se abren en Excel o se revisan a mano, como el resumen mensual oficial (285 KB), y por eso las tablas agregadas de este proyecto se exportan en CSV.

## 4. Perfilamiento

In [9]:
raw.head()

,dt_operacion,aeropuerto_oaci,tipo_operacion,aeropuerto_dgac_orig_dest,aerolinea_dgac,numero_vuelo,actividad_cod,matricula,modelo_avion,modelo_avion_desc,es_internacional,pmd
0,1999-12-31 21:00:00-03:00,SCEL,A,SPIM,LAN,631,U,CCCDM,B763,767-352ER,True,187.00
1,1999-12-31 21:00:00-03:00,SCTC,D,SCUG,6,0,P,CCSPC,C172,172B,False,1.00
2,1999-12-31 21:04:00.001000-03:00,SCEL,A,SCTC,LAN,4248,U,CCCHS,EC35,EC-135 T2,False,2.84
3,1999-12-31 21:08:00-03:00,SCFA,A,SCEL,LCO,2395,U,CCCYK,B732,737-205,False,53.00
4,1999-12-31 21:08:00-03:00,SCFA,A,SCEL,LCO,2394,U,CCCYK,B732,737-205,False,53.00


In [10]:
raw.tail()

,dt_operacion,aeropuerto_oaci,tipo_operacion,aeropuerto_dgac_orig_dest,aerolinea_dgac,numero_vuelo,actividad_cod,matricula,modelo_avion,modelo_avion_desc,es_internacional,pmd
11163786,2026-09-30 20:53:00-03:00,SCSE,D,SCCF,JAT,136,U,CCDIN,A320,A320-271N,False,77.00
11163787,2026-09-30 20:54:00-03:00,SCCF,D,SCEL,LAN,1859,U,CCBEO,A321,A-321-211,False,89.00
11163788,2026-09-30 20:55:00-03:00,SCEL,A,SCQP,LAN,34,U,CCBEI,A321,A-321-211,False,89.00
11163789,2026-09-30 20:57:00-03:00,SCIE,D,SCEL,LAN,10,U,CCBEL,A321,A-321-211,False,89.00
11163790,2026-09-30 20:58:00-03:00,SCCF,A,SCSE,JAT,134,U,CCAWB,A320,A-320-232,False,78.00


In [11]:
raw.sample(5, random_state=42)

,dt_operacion,aeropuerto_oaci,tipo_operacion,aeropuerto_dgac_orig_dest,aerolinea_dgac,numero_vuelo,actividad_cod,matricula,modelo_avion,modelo_avion_desc,es_internacional,pmd
10816381,2026-02-07 23:40:00-03:00,SCEL,A,SCAR,SKU,307,U,CCDBL,A320,A320-251N,False,79.00
1338062,2003-12-30 15:59:00-03:00,SCTB,D,SCTB,6,0,P,CCKIJ,C152,152,False,0.76
2619068,2007-10-14 12:32:00-03:00,SCFA,D,SCFA,6,,P,CCNAC,P28A,PA-28-180 CHERO,False,1.09
596927,2001-11-11 20:23:00-03:00,SCTI,A,SCTB,3R,1,O,CCCRZ,PA38,PA-38-112,False,0.76
5873112,2015-12-08 14:11:00-03:00,SCJO,D,SCJO,21R,0,O,CCCXS,C206,206,False,1.73


In [12]:
print("shape:", raw.shape)
print("columnas:", list(raw.columns))
raw.dtypes

shape: (11163791, 12)
columnas: ['dt_operacion', 'aeropuerto_oaci', 'tipo_operacion', 'aeropuerto_dgac_orig_dest', 'aerolinea_dgac', 'numero_vuelo', 'actividad_cod', 'matricula', 'modelo_avion', 'modelo_avion_desc', 'es_internacional', 'pmd']


dt_operacion                 datetime64[us, America/Santiago]
aeropuerto_oaci                                      category
tipo_operacion                                       category
aeropuerto_dgac_orig_dest                            category
aerolinea_dgac                                       category
numero_vuelo                                         category
actividad_cod                                        category
matricula                                            category
modelo_avion                                         category
modelo_avion_desc                                    category
es_internacional                                         bool
pmd                                                   float64
dtype: object

In [13]:
raw.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 11163791 entries, 0 to 11163790
Data columns (total 12 columns):
 #   Column                     Dtype                           
---  ------                     -----                           
 0   dt_operacion               datetime64[us, America/Santiago]
 1   aeropuerto_oaci            category                        
 2   tipo_operacion             category                        
 3   aeropuerto_dgac_orig_dest  category                        
 4   aerolinea_dgac             category                        
 5   numero_vuelo               category                        
 6   actividad_cod              category                        
 7   matricula                  category                        
 8   modelo_avion               category                        
 9   modelo_avion_desc          category                        
 10  es_internacional           bool                            
 11  pmd                        float64            

In [14]:
raw.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
dt_operacion,11163791,NaN,NaN,NaN,2014-09-10 17:16:20.947549-03:00,1999-12-31 21:00:00-03:00,2008-03-27 19:51:00.000500-03:00,2015-03-28 18:57:00.001000-03:00,2021-09-05 14:33:00.000500-03:00,2026-09-30 20:58:00-03:00,NaN
aeropuerto_oaci,11163791,69,SCEL,3146276,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tipo_operacion,11163791,4,A,5356831,NaN,NaN,NaN,NaN,NaN,NaN,NaN
aeropuerto_dgac_orig_dest,11163791,1671,SCEL,1713457,NaN,NaN,NaN,NaN,NaN,NaN,NaN
aerolinea_dgac,11163791,2568,LAN,1613920,NaN,NaN,NaN,NaN,NaN,NaN,NaN
numero_vuelo,10681646,7442,,1698594,NaN,NaN,NaN,NaN,NaN,NaN,NaN
actividad_cod,11163791,26,U,6157517,NaN,NaN,NaN,NaN,NaN,NaN,NaN
matricula,11163791,13835,CCBAY,48564,NaN,NaN,NaN,NaN,NaN,NaN,NaN
modelo_avion,11163791,1286,A320,1769307,NaN,NaN,NaN,NaN,NaN,NaN,NaN
modelo_avion_desc,11163791,2325,A-321-211,595007,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
raw.nunique().sort_values()

es_internacional                   2
tipo_operacion                     4
actividad_cod                     26
aeropuerto_oaci                   69
pmd                              785
modelo_avion                    1286
aeropuerto_dgac_orig_dest       1671
modelo_avion_desc               2325
aerolinea_dgac                  2568
numero_vuelo                    7442
matricula                      13835
dt_operacion                 6527475
dtype: int64

### 4.1 Perfil construido

In [16]:
TIPO_VARIABLE = {
    "dt_operacion": "temporal", "aeropuerto_oaci": "categórica", "tipo_operacion": "categórica",
    "aeropuerto_dgac_orig_dest": "categórica", "aerolinea_dgac": "categórica", "numero_vuelo": "categórica (código)",
    "actividad_cod": "categórica", "matricula": "categórica (identificador)", "modelo_avion": "categórica",
    "modelo_avion_desc": "textual", "es_internacional": "booleana", "pmd": "numérica",
}
perfil = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "tipo": pd.Series(TIPO_VARIABLE),
    "cardinalidad": raw.nunique(),
    "nulos": raw.isna().sum(),
    "nulos_%": (raw.isna().mean() * 100).round(2),
})
print(f"Registros: {len(raw):,} | Columnas: {raw.shape[1]}")
print("Numéricas:", [c for c, t in TIPO_VARIABLE.items() if t == "numérica"])
print("Temporales:", [c for c, t in TIPO_VARIABLE.items() if t == "temporal"])
print("Categóricas/booleanas:", [c for c, t in TIPO_VARIABLE.items() if t not in ("numérica", "temporal")])
perfil

Registros: 11,163,791 | Columnas: 12
Numéricas: ['pmd']
Temporales: ['dt_operacion']
Categóricas/booleanas: ['aeropuerto_oaci', 'tipo_operacion', 'aeropuerto_dgac_orig_dest', 'aerolinea_dgac', 'numero_vuelo', 'actividad_cod', 'matricula', 'modelo_avion', 'modelo_avion_desc', 'es_internacional']


,dtype,tipo,cardinalidad,nulos,nulos_%
dt_operacion,"datetime64[us, America/Santiago]",temporal,6527475,0,0.00
aeropuerto_oaci,category,categórica,69,0,0.00
tipo_operacion,category,categórica,4,0,0.00
aeropuerto_dgac_orig_dest,category,categórica,1671,0,0.00
aerolinea_dgac,category,categórica,2568,0,0.00
numero_vuelo,category,categórica (código),7442,482145,4.32
actividad_cod,category,categórica,26,0,0.00
matricula,category,categórica (identificador),13835,0,0.00
modelo_avion,category,categórica,1286,0,0.00
modelo_avion_desc,category,textual,2325,0,0.00


## 5. Auditoría de calidad

### 5.1 Ausentes y vacíos

`isna()` no detecta todo: varias columnas usan **cadenas vacías o espacios** como "sin dato".

In [17]:
raw.isna().sum()

dt_operacion                      0
aeropuerto_oaci                   0
tipo_operacion                    0
aeropuerto_dgac_orig_dest         0
aerolinea_dgac                    0
numero_vuelo                 482145
actividad_cod                     0
matricula                         0
modelo_avion                      0
modelo_avion_desc                 0
es_internacional                  0
pmd                          275115
dtype: int64

In [18]:
def vacios(s):
    s = s.dropna().astype("string")
    return pd.Series({
        "vacías_o_espacios": int((s.str.strip() == "").sum()),
        "con_espacios_extremos": int((s != s.str.strip()).sum()),
        "con_minúsculas": int((s != s.str.upper()).sum()),
    })

pd.DataFrame({c: vacios(raw[c]) for c in prep.TEXT_COLS}).T

,vacías_o_espacios,con_espacios_extremos,con_minúsculas
aeropuerto_oaci,0,0,0
tipo_operacion,0,0,1
aeropuerto_dgac_orig_dest,0,0,0
aerolinea_dgac,0,3,17
numero_vuelo,2865981,1698824,3
actividad_cod,0,0,0
matricula,0,4,0
modelo_avion,6,22196,2
modelo_avion_desc,36763,1055,0


In [19]:
nv = raw["numero_vuelo"].astype("string").str.strip()
print("numero_vuelo == '0':", f"{(nv == '0').sum():,}")
print("por actividad (top 5):")
raw.loc[nv == "0", "actividad_cod"].value_counts().head(5)

numero_vuelo == '0': 1,371,917
por actividad (top 5):


actividad_cod
R    392061
P    371779
U    364904
O     83329
N     41519
Name: count, dtype: int64

### 5.2 Duplicados

In [20]:
n_dup = raw.duplicated().sum()
print(f"Duplicados exactos: {n_dup:,} ({n_dup/len(raw):.4%})")
dup = raw[raw.duplicated(keep=False)].sort_values(list(raw.columns))
dup.head(6)

Duplicados exactos: 1,122 (0.0101%)


,dt_operacion,aeropuerto_oaci,tipo_operacion,aeropuerto_dgac_orig_dest,aerolinea_dgac,numero_vuelo,actividad_cod,matricula,modelo_avion,modelo_avion_desc,es_internacional,pmd
10,1999-12-31 21:39:00-03:00,SCFA,A,SCEL,LAN,158,U,CCCJW,B732,737-200,False,53.00
11,1999-12-31 21:39:00-03:00,SCFA,A,SCEL,LAN,158,U,CCCJW,B732,737-200,False,53.00
15,1999-12-31 21:55:00-03:00,SCFA,D,SCDA,LCO,2394,U,CCCYK,B732,737-205,False,53.00
16,1999-12-31 21:55:00-03:00,SCFA,D,SCDA,LCO,2394,U,CCCYK,B732,737-205,False,53.00
508,2000-01-01 21:13:00-03:00,SCFA,D,SCDA,VAT,492,U,CCCSL,B732,737-200,False,54.00
509,2000-01-01 21:13:00-03:00,SCFA,D,SCDA,VAT,492,U,CCCSL,B732,737-200,False,54.00


In [21]:
dup["dt_operacion"].dt.year.value_counts().sort_index()

dt_operacion
1999      4
2000     20
2001     11
2002     71
2003     42
2004    433
2005    340
2006     44
2007    594
2008      8
2009      2
2015      2
Name: count, dtype: int64

### 5.3 Dominios y categorías

In [22]:
raw["tipo_operacion"].value_counts(dropna=False)

tipo_operacion
A    5356831
D    5325353
W     481606
t          1
Name: count, dtype: int64

In [23]:
raw[raw["tipo_operacion"] == "t"]

,dt_operacion,aeropuerto_oaci,tipo_operacion,aeropuerto_dgac_orig_dest,aerolinea_dgac,numero_vuelo,actividad_cod,matricula,modelo_avion,modelo_avion_desc,es_internacional,pmd
6375752,2017-01-16 17:23:00-03:00,SCSN,t,SCSN,6I,1,O,CCAFD,C172,172S,False,1.16


In [24]:
cod_dim = set(dim_act["codigo"])
cod_datos = set(raw["actividad_cod"].astype(str).unique())
print("Códigos en datos sin descripción en la dimensión:", cod_datos - cod_dim)
print("Códigos de la dimensión ausentes en datos:", cod_dim - cod_datos)
print(dim_act[dim_act["codigo"].isin(cod_dim - cod_datos)])

Códigos en datos sin descripción en la dimensión: set()
Códigos de la dimensión ausentes en datos: {'S', '2'}
   codigo descripcion     tipo
18      S   Militares  Militar
26      2    Policial  General


In [25]:
print("Aeródromos informantes sin ficha en dim_aeropuerto:",
      set(raw["aeropuerto_oaci"].astype(str).unique()) - set(dim_aer["oaci"]))
print(dim_aer.loc[dim_aer["oaci"].isin(raw["aeropuerto_oaci"].unique()), "zona_horaria"].value_counts())

Aeródromos informantes sin ficha en dim_aeropuerto: set()
zona_horaria
America/Santiago        63
America/Punta_Arenas     5
Pacific/Easter           1
Name: count, dtype: int64


Coherencia de `es_internacional`: los códigos OACI chilenos empiezan con `SC` y los códigos internos DGAC (aeródromos sin OACI) tienen 3 caracteres.

In [26]:
od = raw["aeropuerto_dgac_orig_dest"].astype("string")
codigo_chileno = od.str.startswith("SC") | (od.str.len() == 3)
pd.crosstab(raw["es_internacional"], codigo_chileno.rename("código chileno"))

código chileno,False,True
es_internacional,,
False,215882,9515035
True,1432874,0


In [27]:
od[~raw["es_internacional"] & ~codigo_chileno].value_counts().head(8)

aeropuerto_dgac_orig_dest
AC66    22491
AC72    17816
AC67    15674
AC69    14320
AC68    12224
AC65     9237
SHSH     8250
ZZZZ     7362
Name: count, dtype: int64[pyarrow]

Ningún vuelo marcado como internacional tiene un código chileno. Los "domésticos con código no chileno" son códigos internos DGAC de 4 caracteres (`AC66`, `AC72`…, aeródromos sin OACI) y `ZZZZ` (convención OACI para "sin código"). No se detecta incoherencia.

### 5.4 Fechas

In [28]:
dt = raw["dt_operacion"]
print("Zona horaria almacenada:", dt.dt.tz)
print("Rango:", dt.min(), "→", dt.max())
print("Rango en UTC:", dt.min().tz_convert("UTC"), "→", dt.max().tz_convert("UTC"))
print("Registros con segundos ≠ 0:", (dt.dt.second != 0).sum())
print(f"Registros con microsegundos ≠ 0: {(dt.dt.microsecond != 0).sum():,}")

Zona horaria almacenada: America/Santiago
Rango: 1999-12-31 21:00:00-03:00 → 2026-09-30 20:58:00-03:00
Rango en UTC: 2000-01-01 00:00:00+00:00 → 2026-09-30 23:58:00+00:00


Registros con segundos ≠ 0: 0


Registros con microsegundos ≠ 0: 1,066,832


- La documentación dice "fecha y hora en UTC", pero el archivo almacena la zona `America/Santiago`. Al ser un *timestamp* con zona, el instante es inequívoco; sólo cambia la representación. El primer registro (`1999-12-31 21:00 -03`) es exactamente `2000-01-01 00:00 UTC`: **los datos se cortan en UTC**, por eso aparecen 3 horas "de 1999" en hora chilena.
- El último registro es `2026-09-30 23:58 UTC`: la documentación indica que se incluye el primer día del mes siguiente, pero en esta versión no ocurre. Las últimas ~3 horas locales de septiembre de 2026 no están.
- 1,07 M registros traen fracciones de segundo (`.001`, `.002`…) aunque los segundos siempre son 0: es un artefacto de carga (probable desempate de registros), no información real.

### 5.5 PMD (peso máximo de despegue, toneladas)

In [29]:
pmd = raw["pmd"]
print(pmd.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))
print(f"\npmd == 0: {(pmd == 0).sum():,}  |  pmd > 650 t: {(pmd > 650).sum():,}")

count   10,888,676.00
mean            45.08
std             61.35
min              0.00
1%               0.55
5%               0.73
25%              1.16
50%              7.00
75%             77.00
95%            185.00
99%            276.00
max          2,100.00
Name: pmd, dtype: float64

pmd == 0: 25,823  |  pmd > 650 t: 1,078


In [30]:
ref_modelo = raw.groupby("modelo_avion", observed=True)["pmd"].agg(["median", "min", "max", "count"])
ref_modelo.sort_values("count", ascending=False).head(10)

,median,min,max,count
modelo_avion,,,,
A320,77.00,0.00,347.00,1696516
C172,1.04,0.00,187.00,996344
B732,54.00,0.00,185.00,927060
A321,89.00,77.00,97.00,614191
A319,70.00,0.00,365.00,541128
C150,0.73,0.00,187.00,356641
PA31,2.95,0.00,187.00,252491
B763,187.00,0.00,500.00,248991
PA38,0.76,0.00,187.00,245345


In [31]:
raw.loc[pmd.nlargest(1000).index].groupby(["modelo_avion", "pmd"], observed=True).size().sort_values(ascending=False).head(8)

modelo_avion  pmd     
T34T          1,340.00    948
AL3           2,100.00     28
C210          1,500.00     20
C206          1,640.00      2
F33A          1,450.00      2
dtype: int64

El PMD se registra por operación y **es inconsistente dentro de un mismo modelo**: un Cessna 172 (≈ 1,1 t) aparece con 187 t, un A320 con 0 t, y un helicóptero Alouette III (≈ 2,2 t) con 2.100 t (probable carga en kg en lugar de toneladas). Un T-34 Turbo Mentor aparece 1.007 veces con 1.340 t. El valor máximo físicamente posible es 640 t (An-225, que efectivamente aparece con ese valor).

### 5.6 Conciliación con el resumen oficial (CSV)

Si la limpieza y la conversión horaria son correctas, agregar la bitácora (sólo A y D, mes en hora local de cada aeródromo) debería reproducir `operaciones-aeropuertos.csv`.

In [32]:
resumen.info()
print("Duplicados de clave (mes, aeródromo, ámbito):",
      resumen.duplicated(["mes_id", "aeropuerto_oaci", "internacional_domestico"]).sum())
resumen.describe()

<class 'pandas.DataFrame'>
RangeIndex: 16172 entries, 0 to 16171
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   mes_id                   16172 non-null  int64
 1   aeropuerto_oaci          16172 non-null  str  
 2   internacional_domestico  16172 non-null  str  
 3   cnt_operaciones          16172 non-null  int64
dtypes: int64(2), str(2)
memory usage: 584.5 KB
Duplicados de clave (mes, aeródromo, ámbito): 0


,mes_id,cnt_operaciones
count,"16,172.00","16,172.00"
mean,"201,362.14",660.54
std,769.77,"1,211.69"
min,"200,001.00",1.00
25%,"200,707.00",23.00
50%,"201,405.00",172.00
75%,"202,008.00",782.00
max,"202,609.00","10,722.00"


In [33]:
tmp = prep.agregar_variables(raw[raw["tipo_operacion"].isin(["A", "D"])][["dt_operacion", "aeropuerto_oaci", "es_internacional", "actividad_cod", "pmd", "aeropuerto_dgac_orig_dest"]],
                             dim_aer, dim_act)
recon = (tmp.assign(internacional_domestico=np.where(tmp["es_internacional"], "I", "D"))
            .groupby(["mes_id", "aeropuerto_oaci", "internacional_domestico"], observed=True).size()
            .rename("bitacora").reset_index())
recon["aeropuerto_oaci"] = recon["aeropuerto_oaci"].astype(str)
conc = resumen.merge(recon, how="outer", on=["mes_id", "aeropuerto_oaci", "internacional_domestico"], indicator=True)
conc["diferencia"] = conc["cnt_operaciones"] - conc["bitacora"]
print(conc["_merge"].value_counts())
print(f"Celdas idénticas: {(conc['diferencia'] == 0).sum():,} de {len(conc):,}")
conc[(conc["diferencia"] != 0) | (conc["_merge"] != "both")]

_merge
both          16172
right_only       11
left_only         0
Name: count, dtype: int64
Celdas idénticas: 16,159 de 16,183


,mes_id,aeropuerto_oaci,internacional_domestico,cnt_operaciones,bitacora,_merge,diferencia
0,199912,SCAR,D,NaN,3,right_only,NaN
1,199912,SCCI,D,NaN,3,right_only,NaN
2,199912,SCDA,D,NaN,5,right_only,NaN
3,199912,SCEL,D,NaN,7,right_only,NaN
4,199912,SCEL,I,NaN,5,right_only,NaN
5,199912,SCFA,D,NaN,8,right_only,NaN
6,199912,SCIE,D,NaN,1,right_only,NaN
7,199912,SCIP,D,NaN,1,right_only,NaN
8,199912,SCTB,D,NaN,1,right_only,NaN
9,199912,SCTC,D,NaN,4,right_only,NaN


In [34]:
del tmp, recon

**Resultado de la conciliación:** 16.159 de 16.172 celdas coinciden exactamente. Todas las diferencias tienen explicación:

- **1999-12** (11 celdas sólo en la bitácora): las primeras 3 horas UTC del 2000 caen en 1999 en hora chilena; el resumen oficial no las publica.
- **2017-01 SCSN** (+1 en el CSV): es el registro con `tipo_operacion = 't'`; el resumen oficial lo contó como operación.
- **2026-09** (CSV mayor): el resumen incluye las últimas horas locales de septiembre que la bitácora no trae.

Además, la columna del CSV se llama `cnt_operaciones`, no `n_operaciones` como dice la documentación. La conciliación confirma que (i) la hora local debe calcularse con la zona de cada aeródromo y (ii) **el resumen oficial incluye los registros duplicados**, por lo que nuestras cifras limpias serán levemente menores (≈ 1.200 operaciones en 26 años).